# Phase 2: Data Cleaning

**Notebook:** `02_clean_documents.ipynb`  
**Matching module:** `03_ingestion/02_clean_documents.py`

**Purpose:** Remove unwanted control characters, extra spaces, standalone page numbers, repeated headers, PDF line-wrap noise, and other non-content formatting while preserving source provenance and healthcare safety wording.

## Input and output contract

**Input:** `01_data/processed/01_loaded_documents.json` from Phase 1.

**Data outputs:** `02_cleaned_documents.json`, `02_cleaning_report.json`, `02_cleaning_audit.csv`, and `02_rejected_documents.json`.

**Plot outputs:** `plots/02_cleaning_operations.png` and `plots/02_characters_removed_by_source_type.png`.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

from IPython.display import Image, display


def find_project_root(start: Path) -> Path:
    """Locate the project from Jupyter or repository execution."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "03_ingestion" / "02_clean_documents.py").is_file():
            return candidate
        nested = candidate / "hospital_patient_helpdesk_chatbot"
        if (nested / "03_ingestion" / "02_clean_documents.py").is_file():
            return nested
    raise FileNotFoundError("Could not locate hospital_patient_helpdesk_chatbot.")


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DATA_DIR = PROJECT_ROOT / "01_data" / "processed"
INPUT_PATH = PROCESSED_DATA_DIR / "01_loaded_documents.json"
MODULE_PATH = PROJECT_ROOT / "03_ingestion" / "02_clean_documents.py"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input file: {INPUT_PATH}")
print(f"Cleaning module: {MODULE_PATH}")


In [ ]:
def load_module(module_path: Path):
    """Load the numbered workflow module from its file path."""
    specification = importlib.util.spec_from_file_location("clean_documents_module", module_path)
    if specification is None or specification.loader is None:
        raise ImportError(f"Could not load module: {module_path}")
    module = importlib.util.module_from_spec(specification)
    sys.modules[specification.name] = module
    specification.loader.exec_module(module)
    return module


cleaning = load_module(MODULE_PATH)
print(f"Minimum cleaned text length: {cleaning.MINIMUM_TEXT_LENGTH}")


## Cleaning rules

1. Normalize Unicode compatibility characters with NFKC.
2. Remove non-printing controls while retaining meaningful punctuation.
3. Remove standalone page-number lines.
4. Remove adjacent duplicate lines and duplicated title prefixes.
5. Repair PDF layout-wrapped lines within paragraph boundaries.
6. Normalize repeated horizontal whitespace and excessive blank lines.
7. Reject only records shorter than the configured threshold after cleaning.

The cleaner does not lowercase text, stem words, alter IDs, or modify source metadata.

In [ ]:
input_documents = cleaning.load_ingested_documents(INPUT_PATH)
pdf_example = next(document for document in input_documents if document["source_type"] == "pdf")
preview_text, preview_operations, preview_metrics = cleaning.clean_text(
    pdf_example["text"],
    repair_pdf_line_wraps=True,
)

print(f"Source: {pdf_example['source_file']}")
print(f"Operations: {preview_operations}")
print(f"Metrics: {preview_metrics}")
print("\nBefore:")
print(pdf_example["text"][:500])
print("\nAfter:")
print(preview_text[:500])


In [ ]:
result = cleaning.run_cleaning(
    input_path=INPUT_PATH,
    output_dir=PROCESSED_DATA_DIR,
)
cleaning.print_result(result)


In [ ]:
cleaned_documents = json.loads(result.cleaned_documents_path.read_text(encoding="utf-8"))
report = json.loads(result.cleaning_report_path.read_text(encoding="utf-8"))
rejected_documents = json.loads(result.rejected_documents_path.read_text(encoding="utf-8"))

if len(cleaned_documents) != report["cleaned_documents"]:
    raise RuntimeError("Cleaned document count does not match the report.")
if len({document["document_id"] for document in cleaned_documents}) != len(cleaned_documents):
    raise RuntimeError("Duplicate document IDs found after cleaning.")
if any(not document["text"].strip() for document in cleaned_documents):
    raise RuntimeError("An empty cleaned document was written.")

print("Cleaning validation passed.")
print(json.dumps(report, indent=2))
print(f"Rejected records: {len(rejected_documents)}")


## Diagnostic plots

The operations chart shows how many documents triggered each cleaning rule. The source-type chart shows where characters were removed, making the transformation auditable and helping detect unexpectedly aggressive cleaning.

In [ ]:
display(Image(filename=str(result.operations_plot_path)))
display(Image(filename=str(result.characters_plot_path)))


In [ ]:
print("Output files:")
for output_name in report["output_files"]:
    output_path = PROCESSED_DATA_DIR / output_name
    print(f"- {output_path}: {output_path.stat().st_size:,} bytes")


## Notebook and Python module roles

The notebook imports the shared module and adds a guided rule explanation, before-and-after preview, report inspection, and inline plot display. The Python module contains reusable cleaning rules, schemas, validation, reporting, plotting, output writing, and command-line behavior for automation.

## Safety and next step

- Cleaning is content-neutral and does not infer diagnoses or alter clinical meaning.
- Source fields and structured metadata remain available for traceability.
- Rejected records are written separately rather than silently deleted.
- Continue with `03_chunk_documents.ipynb` using `02_cleaned_documents.json`.